In [1]:
import os
import getpass
import pandas as pd
from sqlalchemy import URL, create_engine, text

db_password = os.getenv("DB_PASSWORD") or getpass.getpass("MySQL password: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
    database="aggregation_test",
)

engine = create_engine(connection_url, pool_pre_ping=True)

with engine.connect() as conn:
    interaction_sample = pd.read_sql(
        text("""
            SELECT customer_id, product_id, quantity, sale_date
            FROM sales
            LIMIT 10
        """),
        conn,
    )

display(interaction_sample)

,customer_id,product_id,quantity,sale_date
0,1,1,1,2023-01-01
1,2,2,2,2023-01-02
2,3,3,3,2023-01-03
3,4,4,4,2023-01-04
4,5,5,5,2023-01-05
5,6,6,6,2023-01-06
6,7,7,7,2023-01-07
7,8,8,8,2023-01-08
8,9,9,9,2023-01-09
9,10,10,10,2023-01-10


In [2]:
interaction_query = text("""
    SELECT
        customer_id,
        product_id,
        SUM(quantity) AS interaction_strength,
        MAX(sale_date) AS last_interaction
    FROM sales
    GROUP BY customer_id, product_id
    ORDER BY customer_id, product_id
""")

with engine.connect() as conn:
    user_item_interactions = pd.read_sql(interaction_query, conn)

print("Unique customer-product interactions:", len(user_item_interactions))
display(user_item_interactions.head(10))


Unique customer-product interactions: 10000


,customer_id,product_id,interaction_strength,last_interaction
0,1,1,10.0,2026-08-26
1,2,2,20.0,2026-08-27
2,3,3,30.0,2026-08-28
3,4,4,40.0,2026-08-29
4,5,5,50.0,2026-08-30
5,6,6,60.0,2026-08-31
6,7,7,70.0,2026-09-01
7,8,8,80.0,2026-09-02
8,9,9,90.0,2026-09-03
9,10,10,100.0,2026-09-04


In [3]:
target_customer_id = 1

user_based_query = text("""
    WITH user_items AS (
        SELECT
            customer_id,
            product_id,
            SUM(quantity) AS interaction_strength
        FROM sales
        GROUP BY customer_id, product_id
    ),
    target_items AS (
        SELECT product_id
        FROM user_items
        WHERE customer_id = :target_customer_id
    ),
    target_count AS (
        SELECT COUNT(*) AS item_count
        FROM target_items
    ),
    peer_counts AS (
        SELECT
            customer_id,
            COUNT(*) AS item_count
        FROM user_items
        GROUP BY customer_id
    ),
    common_items AS (
        SELECT
            peer.customer_id AS similar_customer_id,
            COUNT(*) AS common_count
        FROM user_items AS peer
        JOIN target_items AS target
          ON peer.product_id = target.product_id
        WHERE peer.customer_id <> :target_customer_id
        GROUP BY peer.customer_id
    ),
    user_similarity AS (
        SELECT
            common.similar_customer_id,
            common.common_count
                / NULLIF(
                    target_count.item_count
                    + peer_counts.item_count
                    - common.common_count,
                    0
                ) AS jaccard_similarity
        FROM common_items AS common
        JOIN peer_counts
          ON peer_counts.customer_id = common.similar_customer_id
        CROSS JOIN target_count
    )
    SELECT
        candidate.product_id,
        ROUND(
            SUM(
                similarity.jaccard_similarity
                * candidate.interaction_strength
            ),
            3
        ) AS recommendation_score,
        COUNT(DISTINCT candidate.customer_id) AS supporting_customers
    FROM user_similarity AS similarity
    JOIN user_items AS candidate
      ON candidate.customer_id = similarity.similar_customer_id
    LEFT JOIN target_items AS already_owned
      ON already_owned.product_id = candidate.product_id
    WHERE already_owned.product_id IS NULL
    GROUP BY candidate.product_id
    ORDER BY recommendation_score DESC, supporting_customers DESC
    LIMIT 10
""")

with engine.connect() as conn:
    user_recommendations = pd.read_sql(
        user_based_query,
        conn,
        params={"target_customer_id": target_customer_id},
    )

print("Recommendations for customer:", target_customer_id)
display(user_recommendations)

Recommendations for customer: 1


,product_id,recommendation_score,supporting_customers


In [4]:
item_based_query = text("""
    WITH user_items AS (
        SELECT
            customer_id,
            product_id,
            SUM(quantity) AS interaction_strength
        FROM sales
        GROUP BY customer_id, product_id
    ),
    target_items AS (
        SELECT product_id
        FROM user_items
        WHERE customer_id = :target_customer_id
    ),
    peer_customers AS (
        SELECT DISTINCT
            peer.customer_id,
            peer.product_id AS matched_target_product
        FROM user_items AS peer
        JOIN target_items AS target
          ON target.product_id = peer.product_id
        WHERE peer.customer_id <> :target_customer_id
    )
    SELECT
        candidate.product_id,
        COUNT(DISTINCT peer.customer_id) AS supporting_customers,
        COUNT(DISTINCT peer.matched_target_product) AS matched_target_products,
        SUM(candidate.interaction_strength) AS total_interaction_strength
    FROM peer_customers AS peer
    JOIN user_items AS candidate
      ON candidate.customer_id = peer.customer_id
    LEFT JOIN target_items AS already_owned
      ON already_owned.product_id = candidate.product_id
    WHERE already_owned.product_id IS NULL
    GROUP BY candidate.product_id
    ORDER BY
        supporting_customers DESC,
        matched_target_products DESC,
        total_interaction_strength DESC
    LIMIT 10
""")

with engine.connect() as conn:
    item_recommendations = pd.read_sql(
        item_based_query,
        conn,
        params={"target_customer_id": target_customer_id},
    )

print("Item-based recommendations for customer:", target_customer_id)
display(item_recommendations)

Item-based recommendations for customer: 1


,product_id,supporting_customers,matched_target_products,total_interaction_strength


In [5]:
popularity_query = text("""
    WITH product_popularity AS (
        SELECT
            product_id,
            COUNT(DISTINCT customer_id) AS customer_count,
            SUM(quantity) AS total_quantity
        FROM sales
        GROUP BY product_id
    ),
    target_items AS (
        SELECT DISTINCT product_id
        FROM sales
        WHERE customer_id = :target_customer_id
    ),
    ranked_products AS (
        SELECT
            popularity.product_id,
            popularity.customer_count,
            popularity.total_quantity,
            ROW_NUMBER() OVER (
                ORDER BY
                    popularity.customer_count DESC,
                    popularity.total_quantity DESC,
                    popularity.product_id
            ) AS recommendation_rank
        FROM product_popularity AS popularity
        LEFT JOIN target_items AS owned
          ON owned.product_id = popularity.product_id
        WHERE owned.product_id IS NULL
    )
    SELECT
        recommendation_rank,
        product_id,
        customer_count,
        total_quantity
    FROM ranked_products
    ORDER BY recommendation_rank
    LIMIT 10
""")

with engine.connect() as conn:
    popularity_recommendations = pd.read_sql(
        popularity_query,
        conn,
        params={"target_customer_id": target_customer_id},
    )

display(popularity_recommendations)

,recommendation_rank,product_id,customer_count,total_quantity
0,1,10,100,10000.0
1,2,20,100,10000.0
2,3,30,100,10000.0
3,4,40,100,10000.0
4,5,50,100,10000.0
5,6,60,100,10000.0
6,7,70,100,10000.0
7,8,80,100,10000.0
8,9,90,100,10000.0
9,10,100,100,10000.0


In [6]:
with engine.connect() as conn:
    purchased_products = set(
        conn.execute(
            text("""
                SELECT DISTINCT product_id
                FROM sales
                WHERE customer_id = :target_customer_id
            """),
            {"target_customer_id": target_customer_id},
        ).scalars().all()
    )

recommendation_sets = {
    "user-based": user_recommendations,
    "item-based": item_recommendations,
    "popularity-based": popularity_recommendations,
}

for method, recommendations in recommendation_sets.items():
    recommended_products = recommendations["product_id"].tolist()

    assert len(recommended_products) == len(set(recommended_products)), (
        f"{method}: duplicate product recommendations found"
    )
    assert not (set(recommended_products) & purchased_products), (
        f"{method}: already purchased products were recommended"
    )

print("Target customer:", target_customer_id)
for method, recommendations in recommendation_sets.items():
    print(f"{method} recommendations:", len(recommendations))

print("TEST CASE 20 RECOMMENDATION VALIDATION PASSED")

Target customer: 1
user-based recommendations: 0
item-based recommendations: 0
popularity-based recommendations: 10
TEST CASE 20 RECOMMENDATION VALIDATION PASSED


In [7]:
recommendation_frames = [
    (
        "user-based",
        user_recommendations,
        "recommendation_score",
    ),
    (
        "item-based",
        item_recommendations,
        "supporting_customers",
    ),
    (
        "popularity-based",
        popularity_recommendations,
        "customer_count",
    ),
]

recommendation_rows = []

for method, frame, score_column in recommendation_frames:
    for rank, (_, row) in enumerate(frame.iterrows(), start=1):
        recommendation_rows.append(
            {
                "customer_id": int(target_customer_id),
                "product_id": int(row["product_id"]),
                "recommendation_method": method,
                "recommendation_score": float(row[score_column]),
                "recommendation_rank": rank,
            }
        )

create_results_table = text("""
    CREATE TABLE IF NOT EXISTS recommendation_results (
        customer_id BIGINT NOT NULL,
        product_id BIGINT NOT NULL,
        recommendation_method VARCHAR(50) NOT NULL,
        recommendation_score DECIMAL(14, 4) NOT NULL,
        recommendation_rank INT NOT NULL,
        generated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (customer_id, product_id, recommendation_method)
    )
""")

with engine.begin() as conn:
    conn.execute(create_results_table)

    conn.execute(
        text("DELETE FROM recommendation_results WHERE customer_id = :customer_id"),
        {"customer_id": target_customer_id},
    )

    if recommendation_rows:
        conn.execute(
            text("""
                INSERT INTO recommendation_results (
                    customer_id,
                    product_id,
                    recommendation_method,
                    recommendation_score,
                    recommendation_rank
                )
                VALUES (
                    :customer_id,
                    :product_id,
                    :recommendation_method,
                    :recommendation_score,
                    :recommendation_rank
                )
            """),
            recommendation_rows,
        )

print("Recommendation rows stored:", len(recommendation_rows))

Recommendation rows stored: 10


In [8]:
with engine.connect() as conn:
    stored_recommendations = pd.read_sql(
        text("""
            SELECT
                customer_id,
                product_id,
                recommendation_method,
                recommendation_score,
                recommendation_rank,
                generated_at
            FROM recommendation_results
            WHERE customer_id = :customer_id
            ORDER BY recommendation_method, recommendation_rank
        """),
        conn,
        params={"customer_id": target_customer_id},
    )

assert len(stored_recommendations) == len(recommendation_rows), (
    f"Expected {len(recommendation_rows)} stored rows, "
    f"found {len(stored_recommendations)}"
)

display(stored_recommendations)

print("Stored recommendation rows verified:", len(stored_recommendations))
print("TEST CASE 20 RECOMMENDATION SYSTEM VALIDATION PASSED")

,customer_id,product_id,recommendation_method,recommendation_score,recommendation_rank,generated_at
0,1,10,popularity-based,100.0,1,2026-11-25 18:09:54
1,1,20,popularity-based,100.0,2,2026-11-25 18:09:54
2,1,30,popularity-based,100.0,3,2026-11-25 18:09:54
3,1,40,popularity-based,100.0,4,2026-11-25 18:09:54
4,1,50,popularity-based,100.0,5,2026-11-25 18:09:54
5,1,60,popularity-based,100.0,6,2026-11-25 18:09:54
6,1,70,popularity-based,100.0,7,2026-11-25 18:09:54
7,1,80,popularity-based,100.0,8,2026-11-25 18:09:54
8,1,90,popularity-based,100.0,9,2026-11-25 18:09:54
9,1,100,popularity-based,100.0,10,2026-11-25 18:09:54


Stored recommendation rows verified: 10
TEST CASE 20 RECOMMENDATION SYSTEM VALIDATION PASSED


In [9]:
with engine.connect() as conn:
    stored_recommendations = pd.read_sql(
        text("""
            SELECT
                customer_id,
                product_id,
                recommendation_method,
                recommendation_score,
                recommendation_rank,
                generated_at
            FROM recommendation_results
            WHERE customer_id = :customer_id
            ORDER BY recommendation_method, recommendation_rank
        """),
        conn,
        params={"customer_id": target_customer_id},
    )

assert len(stored_recommendations) == len(recommendation_rows), (
    f"Expected {len(recommendation_rows)} stored rows, "
    f"found {len(stored_recommendations)}"
)

display(stored_recommendations)

print("Stored recommendation rows verified:", len(stored_recommendations))
print("TEST CASE 20 RECOMMENDATION SYSTEM VALIDATION PASSED")

,customer_id,product_id,recommendation_method,recommendation_score,recommendation_rank,generated_at
0,1,10,popularity-based,100.0,1,2026-11-25 18:09:54
1,1,20,popularity-based,100.0,2,2026-11-25 18:09:54
2,1,30,popularity-based,100.0,3,2026-11-25 18:09:54
3,1,40,popularity-based,100.0,4,2026-11-25 18:09:54
4,1,50,popularity-based,100.0,5,2026-11-25 18:09:54
5,1,60,popularity-based,100.0,6,2026-11-25 18:09:54
6,1,70,popularity-based,100.0,7,2026-11-25 18:09:54
7,1,80,popularity-based,100.0,8,2026-11-25 18:09:54
8,1,90,popularity-based,100.0,9,2026-11-25 18:09:54
9,1,100,popularity-based,100.0,10,2026-11-25 18:09:54


Stored recommendation rows verified: 10
TEST CASE 20 RECOMMENDATION SYSTEM VALIDATION PASSED
